# Actualización de Tablas Vida

## 1. Importación de paqueterías

In [11]:
import pandas as pd
import urllib
from sqlalchemy import create_engine, text


## 2. Conexión a SQL Server

In [12]:

# -----------------------------
# 1️⃣ Conexión a SQL Server Azure
# -----------------------------
params_azure = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=ikaldb.database.windows.net;"
    "DATABASE=actuarial;"
    "UID=CloudSA0052c2f7;"
    "PWD=ricostamales01#;"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
)
engine_azure = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params_azure}",
    fast_executemany=True
)
print("✅ Conexión SQL Server AZURE establecida.")

# -----------------------------
# 2️⃣ Conexión a SQL Server Local
# -----------------------------
params_local = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=IKAL14\\SQLEXPRESS;"
    "DATABASE=actuarial;"
    "Trusted_Connection=yes;"
)
engine_local = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params_local}",
    fast_executemany=True
)
print("✅ Conexión SQL Server Local establecida.")


✅ Conexión SQL Server AZURE establecida.
✅ Conexión SQL Server Local establecida.


## 3. Carga de datos

In [13]:
AñoMes = "202512"  # ← Cambia solo este valor cada mes
archivo_excel = rf"C:/Users/IKAL14/OneDrive - Kot Insurance Company AG/Transporte, Carga y Embarcaciones/{AñoMes[:4]}/{AñoMes}/{AñoMes}_Siniestros_Marine_PROCESADO.xlsx"
df_nuevo = pd.read_excel(archivo_excel)
print(f"✅ Datos leídos: '{archivo_excel}' ({len(df_nuevo):,} registros)")

✅ Datos leídos: 'C:/Users/IKAL14/OneDrive - Kot Insurance Company AG/Transporte, Carga y Embarcaciones/2025/202512/202512_Siniestros_Marine_PROCESADO.xlsx' (4,275 registros)


In [14]:
# -----------------------------Eliminar datos de las tablas de transporte-----------------------------
#Si quieres borrar los datos antiguos de las tablas de transporte, cambia ejecutar a True. Si quieres conservar los datos antiguos, déjalo en False.
def limpiar_tablas_transporte(engine_azure, engine_local, ejecutar=False):
    if not ejecutar:
        print("⏸️ Limpieza desactivada. No se eliminó nada.")
        return

    tablas = ["transporte", "transportehist"]

    for tabla in tablas:
        print(f"🧹 Eliminando datos antiguos de la tabla '{tabla}'...")

        with engine_azure.connect() as conn:
            conn.execute(text(f"TRUNCATE TABLE dbo.{tabla}"))
            conn.commit()

        with engine_local.connect() as conn:
            conn.execute(text(f"TRUNCATE TABLE dbo.{tabla}"))
            conn.commit()

        print(f"✅ Datos antiguos eliminados de la tabla '{tabla}' en ambas bases de datos.")

## 4. Limpieza y carga de tabla transporte

In [15]:
# =============================================================================
# 3️⃣ Actualizar tabla 'transporte' (mes actual)
# =============================================================================
import logging, sys, re
from datetime import datetime

# --- Logger -------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(
            f"etl_transporte_{AñoMes}_{datetime.now():%Y%m%d_%H%M%S}.log",
            encoding="utf-8"
        ),
    ],
)
logger = logging.getLogger("etl.transporte")

# --- Validación de AñoMes ----------------------------------------------------
if not re.match(r"^\d{6}$", str(AñoMes)):
    raise ValueError(f"AñoMes tiene formato inválido: '{AñoMes}'. Se esperaba YYYYMM")
_año_v, _mes_v = int(AñoMes[:4]), int(AñoMes[4:6])
if not (2000 <= _año_v <= 2100) or not (1 <= _mes_v <= 12):
    raise ValueError(f"Período fuera de rango: año={_año_v}, mes={_mes_v}")
logger.info(f"Período validado: {AñoMes}  (año={_año_v}, mes={_mes_v})")

# --- Validación del DataFrame fuente -----------------------------------------
if len(df_nuevo) == 0:
    raise ValueError("El archivo fuente está vacío — se aborta la carga.")
logger.info(f"Registros en archivo fuente: {len(df_nuevo):,}")

# --- Helper: sanitizar nombres de columnas para to_sql -----------------------
# method='multi' usa los nombres de columna como parámetros bind de SQLAlchemy.
# SQL Server/pyodbc no acepta guiones, barras ni caracteres especiales (°, ñ, etc.)
# en nombres de parámetros. Se renombran temporalmente solo para el INSERT;
# las columnas en la tabla SQL se mantienen con sus nombres originales usando
# la cláusula de columnas explícita.
def _safe_col(name: str) -> str:
    """Convierte un nombre de columna a identificador Python/bind válido."""
    safe = re.sub(r"[^A-Za-z0-9_]", "_", name)
    if safe[0].isdigit():
        safe = "c_" + safe
    return safe

def _to_sql_safe(df: pd.DataFrame, table: str, engine, if_exists: str, schema: str = "dbo"):
    """
    Inserta df en table manejando columnas con caracteres especiales.
    Estrategia: renombrar columnas a nombres seguros en el DataFrame,
    luego pasar el mapeo original→seguro al compilador mediante un
    INSERT SELECT desde tabla temporal con alias, o bien usar
    executemany directo sin method='multi'.
    
    La forma más robusta para SQL Server con columnas especiales:
    usar method=None (executemany fila por fila con fast_executemany=True
    ya configurado en el engine) — evita el problema de nombres de bind.
    """
    df.to_sql(
        table,
        con=engine,
        if_exists=if_exists,
        index=False,
        schema=schema,
        chunksize=1000,
        method=None   # executemany estándar; fast_executemany en el engine lo acelera
    )

# --- Actualizar tabla 'transporte' (TRUNCATE + append) -----------------------
logger.info("Actualizando tabla 'transporte' (mes actual)...")

def _upsert_transporte(engine, label: str, df: pd.DataFrame) -> None:
    """TRUNCATE-safe load. Cae en replace si la tabla no existe aún."""
    try:
        with engine.begin() as conn:
            conn.execute(text("TRUNCATE TABLE dbo.transporte"))
        _to_sql_safe(df, "transporte", engine, if_exists="append")
        logger.info(f"[{label}] transporte actualizada (TRUNCATE+append)  ({len(df):,} registros)")
    except Exception as e_trunc:
        if "Invalid object name" in str(e_trunc) or "Cannot truncate" in str(e_trunc):
            logger.warning(f"[{label}] TRUNCATE falló — usando replace (primera carga): {e_trunc}")
            _to_sql_safe(df, "transporte", engine, if_exists="replace")
            logger.info(f"[{label}] transporte creada con replace  ({len(df):,} registros)")
        else:
            raise

_upsert_transporte(engine_azure, "Azure", df_nuevo)
_upsert_transporte(engine_local, "Local", df_nuevo)
logger.info(f"✅ Tabla 'transporte' actualizada con {len(df_nuevo):,} registros")


2026-05-25 19:14:07,738 | INFO | Período validado: 202512  (año=2025, mes=12)
2026-05-25 19:14:07,740 | INFO | Registros en archivo fuente: 4,275
2026-05-25 19:14:07,742 | INFO | Actualizando tabla 'transporte' (mes actual)...
2026-05-25 19:14:15,195 | INFO | [Azure] transporte actualizada (TRUNCATE+append)  (4,275 registros)
2026-05-25 19:14:15,970 | INFO | [Local] transporte actualizada (TRUNCATE+append)  (4,275 registros)
2026-05-25 19:14:15,972 | INFO | ✅ Tabla 'transporte' actualizada con 4,275 registros


## 5. Inserción y actualización de tabla transportehist

In [16]:
# =============================================================================
# 5️⃣ INSERCIÓN EN TABLA TRANSPORTEHIST  (snapshot mensual + columna activo)
# =============================================================================
# Lógica:
#   - transporte      → se reemplaza cada mes con el mes actual
#   - transportehist  → se acumula: cada mes se insertan filas nuevas
#                       activo = 1  →  mes más reciente cargado
#                       activo = 0  →  meses anteriores
#
# NOTA SOBRE COLUMNAS CON CARACTERES ESPECIALES:
#   Columnas como "CLAIMS-ID", "ATTRITIONAL/LARGE", "INWARD POLICY N°",
#   "LoB-Inward" causan DatabaseError cuando se usa method='multi' porque
#   pandas/SQLAlchemy genera parámetros bind con esos nombres y SQL Server
#   los rechaza. Solución: usar method=None (executemany vía fast_executemany)
#   para to_sql, e INSERT...SELECT para transportehist (sin parámetros bind).
# =============================================================================

# =============================================================================
# PASO 1: Obtener estructura de tabla SQL
# =============================================================================
logger.info("=" * 70)
logger.info("PASO 1: Obtener estructura de tabla SQL")
logger.info("=" * 70)

query_columnas = """
SELECT COLUMN_NAME, DATA_TYPE, CHARACTER_MAXIMUM_LENGTH
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_NAME = 'transportehist' AND TABLE_SCHEMA = 'dbo'
ORDER BY ORDINAL_POSITION
"""
df_cols_azure = pd.read_sql(query_columnas, con=engine_azure)
df_cols_local = pd.read_sql(query_columnas, con=engine_local)

# Validar que los esquemas Azure y Local sean idénticos
cols_solo_azure = set(df_cols_azure["COLUMN_NAME"]) - set(df_cols_local["COLUMN_NAME"])
cols_solo_local = set(df_cols_local["COLUMN_NAME"]) - set(df_cols_azure["COLUMN_NAME"])
if cols_solo_azure:
    logger.warning(f"Columnas solo en Azure (no en Local): {sorted(cols_solo_azure)}")
if cols_solo_local:
    logger.warning(f"Columnas solo en Local (no en Azure): {sorted(cols_solo_local)}")
if not cols_solo_azure and not cols_solo_local:
    logger.info("✓ Esquemas Azure y Local son idénticos")

columnas_sql_list  = df_cols_azure["COLUMN_NAME"].tolist()
columnas_sql_upper = {c.upper(): c for c in columnas_sql_list}
columnas_fecha_sql = set(
    df_cols_azure[df_cols_azure["DATA_TYPE"].isin(
        ["date", "datetime", "datetime2"])]["COLUMN_NAME"].tolist()
)

def _get_len_map(df):
    m = (
        df["DATA_TYPE"].isin(["varchar", "nvarchar", "char", "nchar"])
        & df["CHARACTER_MAXIMUM_LENGTH"].notna()
        & (df["CHARACTER_MAXIMUM_LENGTH"] > 0)
    )
    return dict(zip(df.loc[m, "COLUMN_NAME"], df.loc[m, "CHARACTER_MAXIMUM_LENGTH"].astype(int)))

len_azure   = _get_len_map(df_cols_azure)
len_local   = _get_len_map(df_cols_local)
max_len_map = {
    col: min(v for v in [len_azure.get(col), len_local.get(col)] if v is not None)
    for col in set(len_azure) | set(len_local)
}

# Agregar columna activo si no existe
for label, eng in [("Local", engine_local), ("Azure", engine_azure)]:
    with eng.begin() as conn:
        conn.execute(text("""
            IF COL_LENGTH('dbo.transportehist', 'activo') IS NULL
                ALTER TABLE dbo.transportehist ADD [activo] INT NOT NULL DEFAULT 1
        """))
logger.info("✓ Columna [activo] verificada en ambas bases")

# Normalizar nombres de columna (case-insensitive match contra SQL)
rename_map = {
    col: columnas_sql_upper[col.upper()]
    for col in df_nuevo.columns
    if col.upper() in columnas_sql_upper and col != columnas_sql_upper[col.upper()]
}
df_nuevo_renamed = df_nuevo.rename(columns=rename_map)
if rename_map:
    logger.info(f"Columnas renombradas para coincidir con SQL: {rename_map}")

columnas_sql     = set(columnas_sql_list)
columnas_excel   = set(df_nuevo_renamed.columns.tolist())
columnas_validas = sorted(list(columnas_excel & columnas_sql))
logger.info(f"✓ Columnas válidas (intersección Excel∩SQL): {len(columnas_validas)}")

solo_excel = columnas_excel - columnas_sql
if solo_excel:
    logger.warning(f"Columnas solo en Excel (ignoradas): {sorted(solo_excel)}")

solo_sql = columnas_sql - columnas_excel - {"activo"}
if solo_sql:
    logger.warning(f"Columnas solo en SQL (quedarán NULL en el insert): {sorted(solo_sql)}")

# =============================================================================
# PASO 2: Preparar datos del Excel
# =============================================================================
logger.info("=" * 70)
logger.info("PASO 2: Preparar datos del Excel")
logger.info("=" * 70)

df_transporte_limpio = df_nuevo_renamed[columnas_validas].copy()

# Conversión de fechas con log de valores no convertidos
cols_fecha_presentes = [c for c in columnas_fecha_sql if c in df_transporte_limpio.columns]
for col in cols_fecha_presentes:
    ser = df_transporte_limpio[col]
    if pd.api.types.is_datetime64_any_dtype(ser):
        continue
    antes_nulos = ser.isna().sum()
    if pd.api.types.is_integer_dtype(ser) or pd.api.types.is_float_dtype(ser):
        sample = ser.dropna()
        if len(sample) > 0 and not sample.between(1900, 2100).all():
            df_transporte_limpio[col] = pd.to_datetime(
                ser, unit="D", origin="1899-12-30", errors="coerce"
            )
        else:
            logger.info(f"  [{col}] valores en rango año (1900-2100) — se deja como numérico")
            continue
    else:
        df_transporte_limpio[col] = pd.to_datetime(ser, errors="coerce")
    nuevos_nulos = df_transporte_limpio[col].isna().sum() - antes_nulos
    if nuevos_nulos > 0:
        logger.warning(f"  [{col}] conversión de fecha produjo {nuevos_nulos:,} NaT nuevos")
    else:
        logger.info(f"  [{col}] conversión de fecha OK")

# Truncado de strings con log
for col, max_len in max_len_map.items():
    if col in df_transporte_limpio.columns and df_transporte_limpio[col].dtype == object:
        mask_larga = df_transporte_limpio[col].str.len().gt(max_len)
        n_afectados = mask_larga.sum()
        if n_afectados > 0:
            df_transporte_limpio[col] = df_transporte_limpio[col].str[:max_len]
            logger.warning(f"  [{col}] truncada a {max_len} chars en {n_afectados:,} registros")

df_transporte_limpio["activo"] = 1
logger.info(f"✓ {len(df_transporte_limpio):,} registros preparados con activo = 1")

# =============================================================================
# PASO 3: Cargar datos a tabla temporal
# =============================================================================
logger.info("=" * 70)
logger.info("PASO 3: Cargar datos a tabla temporal")
logger.info("=" * 70)

# method=None → usa executemany estándar de pyodbc con fast_executemany=True
# (configurado en el engine). Esto evita el error de parámetros bind con
# caracteres especiales que ocurre con method='multi'.
def _load_temp(engine, label, df):
    try:
        with engine.begin() as conn:
            conn.execute(text("TRUNCATE TABLE dbo.temp_transportehist_mes"))
        df.to_sql(
            "temp_transportehist_mes", con=engine, if_exists="append",
            index=False, schema="dbo", chunksize=1000, method=None
        )
        logger.info(f"[{label}] Tabla temporal cargada con TRUNCATE+append ({len(df):,} registros)")
    except Exception:
        df.to_sql(
            "temp_transportehist_mes", con=engine, if_exists="replace",
            index=False, schema="dbo", chunksize=1000, method=None
        )
        logger.info(f"[{label}] Tabla temporal creada con replace ({len(df):,} registros)")

_load_temp(engine_local, "Local", df_transporte_limpio)
_load_temp(engine_azure, "Azure", df_transporte_limpio)

# Detectar y corregir mismatches de tipo DATE vs bigint
query_mismatch = """
SELECT t.COLUMN_NAME, t.DATA_TYPE AS tipo_hist, s.DATA_TYPE AS tipo_temp
FROM INFORMATION_SCHEMA.COLUMNS t
JOIN INFORMATION_SCHEMA.COLUMNS s
    ON t.COLUMN_NAME = s.COLUMN_NAME
   AND s.TABLE_NAME = 'temp_transportehist_mes' AND s.TABLE_SCHEMA = 'dbo'
WHERE t.TABLE_NAME = 'transportehist' AND t.TABLE_SCHEMA = 'dbo'
  AND t.DATA_TYPE <> s.DATA_TYPE
ORDER BY t.COLUMN_NAME
"""
mismatches = pd.read_sql(query_mismatch, con=engine_local)
if not mismatches.empty:
    logger.warning(f"Mismatches de tipo detectados:\n{mismatches.to_string(index=False)}")

cols_bigint_date = mismatches[
    (mismatches["tipo_hist"] == "date") & (mismatches["tipo_temp"] == "bigint")
]["COLUMN_NAME"].tolist()

if cols_bigint_date:
    logger.info(f"Migrando columnas DATE→INT en transportehist: {cols_bigint_date}")
    for col in cols_bigint_date:
        if not re.match(r"^[\w\s\-]+$", col):
            logger.error(f"  Nombre de columna no seguro, se omite: '{col}'")
            continue
        col_tmp = re.sub(r"[\s\-]+", "_", col) + "_yr_tmp"
        for label, eng in [("Local", engine_local), ("Azure", engine_azure)]:
            with eng.begin() as conn:
                conn.execute(text(
                    f"IF COL_LENGTH('dbo.transportehist','{col_tmp}') IS NOT NULL "
                    f"ALTER TABLE dbo.transportehist DROP COLUMN [{col_tmp}]"
                ))
                conn.execute(text(f"ALTER TABLE dbo.transportehist ADD [{col_tmp}] INT NULL"))
                conn.execute(text(f"UPDATE dbo.transportehist SET [{col_tmp}] = YEAR([{col}])"))
                conn.execute(text(f"""
                    DECLARE @sql NVARCHAR(MAX) = ''
                    SELECT @sql = @sql + 'DROP INDEX [' + i.name + '] ON dbo.transportehist; '
                    FROM sys.indexes i
                    JOIN sys.index_columns ic ON i.object_id=ic.object_id AND i.index_id=ic.index_id
                    JOIN sys.columns c ON ic.object_id=c.object_id AND ic.column_id=c.column_id
                    WHERE OBJECT_NAME(i.object_id)='transportehist' AND c.name='{col}'
                      AND i.is_primary_key=0 AND i.is_unique_constraint=0
                    IF LEN(@sql) > 0 EXEC(@sql)
                """))
                conn.execute(text(f"ALTER TABLE dbo.transportehist DROP COLUMN [{col}]"))
                conn.execute(text(
                    f"EXEC sp_rename @objname=N'dbo.transportehist.{col_tmp}', "
                    f"@newname=N'{col}', @objtype=N'COLUMN'"
                ))
        logger.info(f"  ✓ [{col}] migrada a INT")

# =============================================================================
# PASO 4: Carga atómica en transportehist (una transacción por engine)
# =============================================================================
logger.info("=" * 70)
logger.info("PASO 4: Carga atómica en transportehist")
logger.info("=" * 70)

# Rango de fechas sargable (usa índice)
fecha_inicio  = f"{_año_v}-{_mes_v:02d}-01"
_mes_sig      = _mes_v + 1 if _mes_v < 12 else 1
_año_sig      = _año_v  if _mes_v < 12 else _año_v + 1
fecha_fin_exc = f"{_año_sig}-{_mes_sig:02d}-01"

# INSERT...SELECT desde tabla temporal — sin parámetros bind, evita el problema
# de caracteres especiales en nombres de columna. Los nombres van entre corchetes.
all_cols     = columnas_validas + ["activo"]
cols_sql_str = ", ".join([f"[{c}]" for c in all_cols])
insert_sql   = f"""
    INSERT INTO dbo.transportehist ({cols_sql_str})
    SELECT {cols_sql_str} FROM dbo.temp_transportehist_mes
"""

def _cargar_mes_atomico(engine, label: str) -> dict:
    """
    Ejecuta DELETE + UPDATE activo + INSERT en una sola transacción.
    Si cualquier paso falla → ROLLBACK automático (eng.begin()).
    """
    with engine.begin() as conn:
        # 1. Conteo antes
        antes = conn.execute(
            text("SELECT COUNT(*) FROM dbo.transportehist")
        ).scalar()

        # 2. Eliminar mes si ya existe (idempotencia) — rango sargable
        deleted = conn.execute(text("""
            DELETE FROM dbo.transportehist
            WHERE [MES CARGA] >= :fi AND [MES CARGA] < :ff
        """), {"fi": fecha_inicio, "ff": fecha_fin_exc}).rowcount
        if deleted:
            logger.info(f"[{label}] DELETE: {deleted:,} registros del período {AñoMes} eliminados")

        # 3. Desactivar meses anteriores
        deactivated = conn.execute(
            text("UPDATE dbo.transportehist SET [activo] = 0 WHERE [activo] = 1")
        ).rowcount
        logger.info(f"[{label}] Desactivados: {deactivated:,} registros → activo=0")

        # 4. Insertar nuevo mes desde tabla temporal (INSERT...SELECT, sin bind params)
        inserted = conn.execute(text(insert_sql)).rowcount
        logger.info(f"[{label}] Insertados: {inserted:,} registros → activo=1")

        # 5. Validación de integridad dentro de la misma transacción
        meses_activos = conn.execute(text("""
            SELECT COUNT(DISTINCT
                CONCAT(YEAR([MES CARGA]), '-',
                       RIGHT('0' + CAST(MONTH([MES CARGA]) AS VARCHAR), 2))
            )
            FROM dbo.transportehist
            WHERE [activo] = 1
        """)).scalar()
        if meses_activos != 1:
            raise RuntimeError(
                f"[{label}] Estado inválido: {meses_activos} meses con activo=1 → ROLLBACK"
            )

        despues = antes - deleted + inserted
        logger.info(
            f"[{label}] ✅ Transacción OK — "
            f"antes={antes:,}  deleted={deleted:,}  inserted={inserted:,}  "
            f"esperado_despues={despues:,}"
        )
        return {"label": label, "antes": antes, "deleted": deleted,
                "deactivated": deactivated, "inserted": inserted, "despues": despues}

resultados = []
for label, eng in [("Local", engine_local), ("Azure", engine_azure)]:
    try:
        res = _cargar_mes_atomico(eng, label)
        resultados.append(res)
    except Exception as e:
        logger.error(f"[{label}] FALLO — ROLLBACK automático. Error: {e}", exc_info=True)
        raise

# =============================================================================
# PASO 5: Resumen y limpieza
# =============================================================================
logger.info("=" * 70)
logger.info("RESUMEN FINAL")
logger.info("=" * 70)

try:
    resumen = pd.read_sql("""
        SELECT [activo], COUNT(*) AS registros
        FROM dbo.transportehist
        GROUP BY [activo]
        ORDER BY [activo] DESC
    """, con=engine_azure)

    logger.info("transportehist Azure — distribución por activo:")
    for _, row in resumen.iterrows():
        estado = f"MES ACTIVO ({AñoMes})" if row["activo"] == 1 else "Histórico"
        logger.info(
            f"   activo={int(row['activo'])}  →  "
            f"{int(row['registros']):,} registros  ({estado})"
        )
finally:
    for eng, label in [(engine_local, "Local"), (engine_azure, "Azure")]:
        try:
            with eng.begin() as conn:
                conn.execute(text(
                    "IF OBJECT_ID('dbo.temp_transportehist_mes','U') IS NOT NULL "
                    "DROP TABLE dbo.temp_transportehist_mes"
                ))
            logger.info(f"🧹 Tabla temporal eliminada en {label}")
        except Exception as e_drop:
            logger.warning(f"No se pudo eliminar tabla temporal en {label}: {e_drop}")

logger.info("=" * 70)


2026-05-25 19:14:16,044 | INFO | ======================================================================
2026-05-25 19:14:16,047 | INFO | PASO 1: Obtener estructura de tabla SQL
2026-05-25 19:14:16,049 | INFO | ======================================================================
2026-05-25 19:14:16,348 | INFO | ✓ Esquemas Azure y Local son idénticos
2026-05-25 19:14:16,471 | INFO | ✓ Columna [activo] verificada en ambas bases
2026-05-25 19:14:16,477 | INFO | Columnas renombradas para coincidir con SQL: {'StandRe LoB': 'STANDRE LOB', 'StandRe detailed LoB': 'STANDRE DETAILED LOB', 'LoB-Inward': 'LOB-INWARD', 'Cumulative SALVAGE': 'CUMULATIVE SALVAGE', 'Cumulative EXPENSES PAID': 'CUMULATIVE EXPENSES PAID', 'Cumulative VALUATION EXPENSES': 'CUMULATIVE VALUATION EXPENSES', 'Cumulative CLAIMS PAID': 'CUMULATIVE CLAIMS PAID', 'Total Claims Paid Inward': 'TOTAL CLAIMS PAID INWARD', 'Total Claims Paid no Alae': 'TOTAL CLAIMS PAID NO ALAE', 'OSLR Inward': 'OSLR INWARD', 'Inward Incurred': 'IN